# Ladybug nightly + Icebug: Karate Club graph analytics from icebug-disk

This notebook demonstrates loading a graph stored in **icebug-disk** format (Parquet files) into both Ladybug and Icebug.

The dataset is the Zachary Karate Club graph — 34 nodes, 78 undirected edges stored as 156 directed CSR entries.

The full pipeline: plain DuckDB tables → `icebug-format` CLI → icebug-disk Parquet → Ladybug Cypher + Icebug graph algorithms.

## Environment

Dependencies are declared in `pyproject.toml`. From a fresh checkout, start Jupyter with:

```bash
uv run jupyter notebook ladybug_icebug_disk_karate.ipynb
```

Or execute the notebook from the CLI with:

```bash
uv run jupyter nbconvert --to notebook --execute ladybug_icebug_disk_karate.ipynb --inplace
```

In [1]:
# Optional setup cell: sync dependencies from pyproject.toml.
# If you launched with `uv run jupyter notebook`, this is already done.
!uv sync

Resolved 105 packages in 1ms
Audited 100 packages in 1ms


In [2]:
import ladybug
import icebug as ib
import pyarrow as pa
import pyarrow.parquet as pq
from icebug_format import IcebugMemGraph

print("ladybug", ladybug.__version__)
print("icebug", ib.__version__)


ladybug 0.17.0
icebug 12.7


## Create raw graph tables with DuckDB

Start with plain relational tables — a `nodes` table and an `edges` table. This is the normal input format that `icebug-format` expects:

- `nodes`: any columns; first column is treated as the primary key
- `edges`: must have `source` and `target` columns referencing node primary keys; any extra columns become edge properties

In [3]:
import os, duckdb

SOURCE_DB = "/tmp/karate_notebook/karate.duckdb"
os.makedirs(os.path.dirname(SOURCE_DB), exist_ok=True)

con = duckdb.connect(SOURCE_DB)
con.execute("DROP TABLE IF EXISTS nodes; DROP TABLE IF EXISTS edges")

con.execute("""
    CREATE TABLE nodes AS
    SELECT * FROM (VALUES
        (3,'Mr. Hi'),(14,'Mr. Hi'),(19,'Officer'),(30,'Mr. Hi'),(41,'Mr. Hi'),
        (43,'Mr. Hi'),(54,'Mr. Hi'),(74,'Officer'),(75,'Mr. Hi'),(80,'Mr. Hi'),
        (87,'Officer'),(96,'Mr. Hi'),(104,'Officer'),(126,'Officer'),(143,'Officer'),
        (150,'Officer'),(161,'Officer'),(167,'Officer'),(174,'Mr. Hi'),(183,'Officer'),
        (194,'Officer'),(211,'Mr. Hi'),(218,'Mr. Hi'),(228,'Officer'),(236,'Officer'),
        (266,'Officer'),(268,'Mr. Hi'),(282,'Officer'),(302,'Mr. Hi'),(305,'Officer'),
        (306,'Mr. Hi'),(308,'Mr. Hi'),(318,'Officer'),(333,'Mr. Hi')
    ) t(id, club)
""")

con.execute("""
    CREATE TABLE edges AS
    SELECT * FROM (VALUES
        (19,74),(19,87),(19,104),(19,126),(19,167),(43,87),(54,87),(74,318),
        (75,87),(75,150),(75,167),(80,3),(80,14),(80,41),(80,43),(80,54),
        (80,75),(80,96),(80,174),(80,211),(80,218),(80,268),(80,302),(80,306),
        (80,308),(80,318),(80,333),(96,41),(96,268),(104,87),(104,167),(126,87),
        (143,87),(143,167),(150,87),(150,167),(161,87),(161,167),(167,87),
        (174,54),(174,211),(174,306),(183,74),(183,126),(183,318),(194,87),
        (228,87),(228,167),(236,87),(236,104),(266,87),(266,167),(268,30),
        (282,87),(282,318),(302,30),(302,41),(302,268),(305,87),(305,167),
        (308,3),(308,14),(308,43),(308,54),(308,150),(308,174),(308,306),
        (308,333),(318,87),(318,167),(333,54),(333,75),(333,126),(333,167),
        (333,174),(333,194),(333,282),(333,306)
    ) t(source, target)
""")

print("nodes:", con.execute("SELECT count(*) FROM nodes").fetchone()[0])
print("edges:", con.execute("SELECT count(*) FROM edges").fetchone()[0])
print()
print(con.execute("SELECT * FROM nodes LIMIT 5").df())
print()
print(con.execute("SELECT * FROM edges LIMIT 5").df())
con.close()

nodes: 34
edges: 78

   id     club
0   3   Mr. Hi
1  14   Mr. Hi
2  19  Officer
3  30   Mr. Hi
4  41   Mr. Hi

   source  target
0      19      74
1      19      87
2      19     104
3      19     126
4      19     167


## Convert to icebug-disk with the icebug-format CLI

`icebug-format` reads the DuckDB source, computes CSR adjacency structures, and writes three Parquet files plus a `schema.cypher` to a sibling directory named after `--output-db`.

The `--undirected` flag duplicates each edge in both directions so the CSR encodes a symmetric adjacency — matching the karate club's undirected topology.

In [4]:
OUTPUT_DB  = "/tmp/karate_notebook/karate_csr.duckdb"
DISK_PATH  = OUTPUT_DB.removesuffix(".duckdb")   # /tmp/karate_notebook/karate_csr

!icebug-format \
    --source-db {SOURCE_DB} \
    --output-db {OUTPUT_DB} \
    --add-reverse-edges

import os
print("\nGenerated files:")
for f in sorted(os.listdir(DISK_PATH)):
    print(" ", f)

=== DuckDB to CSR Format Converter ===

Creating CSR graph on FULL DATASET
Source database: /tmp/karate_notebook/karate.duckdb
CSR output database: /tmp/karate_notebook/karate_csr.duckdb
CSR table prefix: karate
Add reverse edges: True
DuckDB memory limit: 26.7GB
Storage path: ./karate_csr

=== Creating CSR Graph Data (Optimized SQL Approach) ===
Step 0: Loading edges and nodes from original DB into new DB...
Discovered node tables: ['nodes']
Discovered edge tables: ['edges']
Node type to table mapping: {'nodes': 'nodes'}
  Copied node table: nodes -> karate_nodes

Step 1: Building per-edge-table CSR structures...

  Processing edges: using fallback node table nodes
    Edges: 156
    indptr: 35 entries
    indices: 156 entries

✓ Built CSR format: 34 nodes, 156 edges across 1 edge types
✓ Saved CSR graph data to /tmp/karate_notebook/karate_csr.duckdb

=== Exporting to Parquet and Generating schema.cypher ===
Parquet output directory: /tmp/karate_notebook/karate_csr
  Exported: karate_

## Load icebug-disk Parquet files → Arrow

The icebug-disk format stores a pre-computed CSR graph as three Parquet files:

- `nodes_<label>.parquet` — node properties
- `indices_<rel>.parquet` — CSR adjacency list (`target` column, 0-indexed row positions)
- `indptr_<rel>.parquet` — CSR row-pointer array (`ptr` column, length = `n_nodes + 1`)

Reading them with `pyarrow.parquet.read_table` yields Arrow tables that can be handed directly to both Ladybug and Icebug without any further conversion.

In [5]:
# Convert parquet → Arrow
nodes   = pq.read_table(f"{DISK_PATH}/nodes_nodes.parquet")
indices = pq.read_table(f"{DISK_PATH}/indices_edges.parquet")
indptr  = pq.read_table(f"{DISK_PATH}/indptr_edges.parquet")

print("nodes schema  :", nodes.schema)
print("indices schema:", indices.schema)
print("indptr schema :", indptr.schema)
print()
print(f"n_nodes={nodes.num_rows}, n_edges={indices.num_rows}, indptr_rows={indptr.num_rows}")
nodes, indices, indptr

nodes schema  : id: int32
club: string
-- schema metadata --
icebug_disk_version: 'v1'
indices schema: target: uint64
-- schema metadata --
icebug_disk_version: 'v1'
indptr schema : ptr: uint64
-- schema metadata --
icebug_disk_version: 'v1'

n_nodes=34, n_edges=156, indptr_rows=35


(pyarrow.Table
 id: int32
 club: string
 ----
 id: [[3,14,19,30,41,...,305,306,308,318,333]]
 club: [["Mr. Hi","Mr. Hi","Officer","Mr. Hi","Mr. Hi",...,"Officer","Mr. Hi","Mr. Hi","Officer","Mr. Hi"]],
 pyarrow.Table
 target: uint64
 ----
 target: [[9,31,9,31,7,...,18,20,27,30,31]],
 pyarrow.Table
 ptr: uint64
 ----
 ptr: [[0,2,4,9,11,...,127,131,140,146,156]])

## Ladybug: native icebug-disk loading

Ladybug understands the icebug-disk format directly via a Cypher extension:

```cypher
CREATE NODE TABLE nodes(...) WITH (storage = '<path>', format = 'icebug-disk')
CREATE REL TABLE edges(FROM nodes TO nodes) WITH (storage = '<path>', format = 'icebug-disk')
```

This is the same schema that `icebug-format` writes into `schema.cypher` alongside the Parquet files. Ladybug scans the Parquet files in-place — no Arrow conversion or manual CSR wiring needed.

In [6]:
disk_db = ladybug.Database()
disk_conn = ladybug.Connection(disk_db)

disk_conn.execute(f"""
    CREATE NODE TABLE nodes(id INT64, club STRING, PRIMARY KEY(id))
    WITH (storage = '{DISK_PATH}', format = 'icebug-disk')
""")
disk_conn.execute(f"""
    CREATE REL TABLE edges(FROM nodes TO nodes)
    WITH (storage = '{DISK_PATH}', format = 'icebug-disk')
""")

disk_conn.query_as_arrow("""
MATCH (a:nodes)-[:edges]->(b:nodes)
RETURN a.id AS node_id, a.club AS club, count(DISTINCT b) AS degree
ORDER BY degree DESC, node_id
LIMIT 10
""", 4096).get_as_arrow().to_pandas()

,node_id,club,degree
0,87,Officer,17
1,80,Mr. Hi,16
2,167,Officer,12
3,333,Mr. Hi,10
4,308,Mr. Hi,9
5,174,Mr. Hi,6
6,318,Officer,6
7,19,Officer,5
8,54,Mr. Hi,5
9,75,Mr. Hi,5


In [7]:
disk_conn.query_as_arrow("""
MATCH (a:nodes)-[:edges]->(b:nodes)
RETURN a.club AS club, count(DISTINCT a) AS members, count(b) AS total_connections
ORDER BY members DESC
""", 4096).get_as_arrow().to_pandas()

,club,members,total_connections
0,Mr. Hi,17,0
1,Officer,17,0


## Ladybug: Arrow CSR registration

The same graph can also be registered with Ladybug directly from the Arrow tables loaded above — useful when the Arrow tables are already in memory (e.g. after building `IcebugMemGraph`). The indices table needs a `to` destination column, so we rename `target` → `to` before passing it.

In [9]:
# Ladybug CSR layout requires the destination column to be named 'to'
db = ladybug.Database()
conn = ladybug.Connection(db)
conn.create_arrow_table("Node", nodes)
conn.create_arrow_rel_table(
    "KNOWS", indices, "Node", "Node",
    "CSR", indptr, "target"
)

conn.query_as_arrow("""
MATCH (a:Node)-[:KNOWS]->(b:Node)
RETURN a.id AS node_id, a.club AS club, count(DISTINCT b) AS degree
ORDER BY degree DESC, node_id
LIMIT 10
""", 4096).get_as_arrow().to_pandas()

,node_id,club,degree
0,87,Officer,17
1,80,Mr. Hi,16
2,167,Officer,12
3,333,Mr. Hi,10
4,308,Mr. Hi,9
5,174,Mr. Hi,6
6,318,Officer,6
7,19,Officer,5
8,54,Mr. Hi,5
9,75,Mr. Hi,5


In [10]:
conn.query_as_arrow("""
MATCH (a:Node)-[:KNOWS]->(b:Node)
RETURN a.club AS club, count(DISTINCT a) AS members, count(b) AS total_connections
ORDER BY members DESC
""", 4096).get_as_arrow().to_pandas()

,club,members,total_connections
0,Mr. Hi,17,0
1,Officer,17,0


## Build IcebugMemGraph from Arrow

The Parquet files already encode a complete CSR structure, so we construct `IcebugMemGraph` directly from the Arrow tables — no recomputation needed.

In [11]:
icemem_graph = IcebugMemGraph(
    src=nodes,
    dest=nodes,
    indices=indices,
    indptr=indptr,
)

print(f"n_nodes={icemem_graph.src.num_rows}")
print(f"n_edges (CSR entries)={icemem_graph.indices.num_rows}")
print(f"indptr length={icemem_graph.indptr.num_rows}")

n_nodes=34
n_edges (CSR entries)=156
indptr length=35


## Icebug: undirected graph — Leiden community detection

The icebug-disk CSR stores both `(u, v)` and `(v, u)` for every undirected edge. Passing `directed=False` to `fromIcebugMemGraph` lets Icebug interpret the doubly-stored adjacency as a proper undirected graph.

In [12]:
undirected_graph = ib.Graph.fromIcebugMemGraph(icemem_graph, directed=False)

leiden = ib.community.ParallelLeidenView(undirected_graph, iterations=4, gamma=1.0, randomize=False)
leiden.run()
partition = leiden.getPartition()
modularity = ib.community.Modularity().getQuality(partition, undirected_graph)

nodes_df = conn.query_as_arrow(
    "MATCH (n:Node) RETURN n.rowid AS rowid, n.id AS id, n.club AS club ORDER BY n.rowid",
    4096,
).get_as_arrow().to_pandas()

communities = nodes_df.copy()
communities["community"] = [partition.subsetOf(i) for i in range(undirected_graph.numberOfNodes())]
communities = communities.sort_values(["community", "club", "id"]).reset_index(drop=True)

print(f"Icebug graph: {undirected_graph.numberOfNodes()} nodes, {undirected_graph.numberOfEdges()} undirected edges")
print(f"Leiden found {partition.numberOfSubsets()} communities; modularity Q={modularity:.4f}")
communities

Icebug graph: 34 nodes, 78 undirected edges
Leiden found 4 communities; modularity Q=0.4198


,rowid,id,club,community
0,0,3,Mr. Hi,0
1,1,14,Mr. Hi,0
2,5,43,Mr. Hi,0
3,6,54,Mr. Hi,0
4,9,80,Mr. Hi,0
5,18,174,Mr. Hi,0
6,21,211,Mr. Hi,0
7,22,218,Mr. Hi,0
8,30,306,Mr. Hi,0
9,31,308,Mr. Hi,0


In [13]:
communities.groupby(["community", "club"]).size().rename("members").reset_index()\
    .sort_values(["community", "members"], ascending=[True, False])

,community,club,members
0,0,Mr. Hi,11
1,1,Officer,6
2,2,Mr. Hi,5
4,3,Officer,11
3,3,Mr. Hi,1


## Icebug: directed graph — PageRank

With `directed=True` (the default), `fromIcebugMemGraph` builds a directed graph that respects edge direction. PageRank measures the relative importance of each node based on incoming link structure.

In [14]:
directed_graph = ib.Graph.fromIcebugMemGraph(icemem_graph)

pr = ib.centrality.PageRank(directed_graph)
pr.run()
pr_scores = pr.scores()

nodes_df = conn.query_as_arrow(
    "MATCH (n:Node) RETURN n.rowid AS rowid, n.id AS id, n.club AS club ORDER BY n.rowid",
    4096,
).get_as_arrow().to_pandas()

nodes_df["pagerank"] = pr_scores
nodes_df.sort_values("pagerank", ascending=False).reset_index(drop=True)

,rowid,id,club,pagerank
0,0,3,Mr. Hi,0.029412
1,1,14,Mr. Hi,0.029412
2,2,19,Officer,0.029412
3,3,30,Mr. Hi,0.029412
4,4,41,Mr. Hi,0.029412
5,5,43,Mr. Hi,0.029412
6,6,54,Mr. Hi,0.029412
7,7,74,Officer,0.029412
8,8,75,Mr. Hi,0.029412
9,9,80,Mr. Hi,0.029412


## What happened

1. Plain `nodes` and `edges` DuckDB tables were created in-memory — the normal relational starting point.
2. `icebug-format --source-db ... --output-db ... --undirected` read those tables, computed CSR adjacency structures, and wrote `nodes_nodes.parquet`, `indices_edges.parquet`, `indptr_edges.parquet`, and `schema.cypher` to disk.
3. **Ladybug native path**: `conn.execute("CREATE ... WITH (storage=..., format='icebug-disk')")` mounted the Parquet files directly — no Python-side conversion or manual CSR wiring needed. This is the simplest way to run Cypher over an icebug-disk dataset.
4. **Arrow path**: `pyarrow.parquet.read_table` converted each Parquet file to an Arrow table. Ladybug also accepts these via `create_arrow_rel_table(..., layout="CSR")`, useful when the Arrow tables are already in memory for downstream use.
5. `IcebugMemGraph` was constructed directly from the Arrow tables — no CSR recomputation needed since the Parquet files already store the CSR structure.
6. **Icebug (undirected)**: `Graph.fromIcebugMemGraph(icemem_graph, directed=False)` interpreted the doubly-stored adjacency as an undirected graph and ran Leiden community detection, recovering the two known social factions.
7. **Icebug (directed)**: `Graph.fromIcebugMemGraph(icemem_graph)` preserved edge direction and PageRank identified the most influential nodes by incoming connectivity.
